In [6]:
import numpy as np
import networkx as nx
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize
import time

# --- 1. Problem Setup ---
seed = np.random.seed(42)
G = nx.random_regular_graph(3, 8, seed)
n_qubits = G.number_of_nodes()
edges = G.edges()
num_edges = len(edges)

# Calculate Max Cut via Brute Force for the Ratio denominator
def get_max_cut(n, edges):
    max_c = 0
    for i in range(2**n):
        b = format(i, f'0{n}b')
        cut = sum(1 for u, v in edges if b[u] != b[v])
        if cut > max_c: max_c = cut
    return max_c

MAX_CUT_VAL = get_max_cut(n_qubits, edges)

# Define Cost Hamiltonian H_C = sum(ZiZj)
pauli_list = []
for i, j in edges:
    p_str = ["I"] * n_qubits
    p_str[n_qubits - 1 - i] = "Z"
    p_str[n_qubits - 1 - j] = "Z"
    pauli_list.append("".join(p_str))
cost_h = SparsePauliOp(pauli_list, coeffs=[1.0] * num_edges)

# --- 2. Adaptive Reflection Logic ---
def add_reflection(qc, state_vec, beta):
    n = qc.num_qubits
    v_gate = QuantumCircuit(n)
    v_gate.prepare_state(state_vec)
    
    # Unitary: V * (I - (1-e^-ib)|0><0|) * V_adj
    qc.append(v_gate.inverse(), range(n))
    qc.x(range(n))
    qc.mcp(-beta, list(range(n-1)), n-1)
    qc.x(range(n))
    qc.append(v_gate, range(n))

# --- 3. Iterative Optimization Loop ---
p_total = 10  # Number of iterative layers
estimator = StatevectorEstimator()
current_state = Statevector.from_label('+' * n_qubits)
ratios = []

print(f"{'Layer':<6} | {'Energy':<10} | {'Exp. Cut':<10} | {'Ratio (%)':<10}")
print("-" * 45)
start_time = time.perf_counter()
for p in range(1, p_total + 1):
    
    def objective(params):
        gamma, beta = params
        qc = QuantumCircuit(n_qubits)
        qc.prepare_state(current_state)
        
        # Phase Separation (Cost)
        for i, j in edges:
            qc.rzz(2 * gamma, i, j)
            
        # Adaptive Mixing (Projector Mixer)
        add_reflection(qc, current_state, beta)
        
        pub = (qc, cost_h)
        return estimator.run([pub]).result()[0].data.evs

    # Layer-wise optimization
    res = minimize(objective, [0.71, 0.39], method='COBYLA')
    
    # --- Calculate Ratio ---
    # Expected Cut = 0.5 * (num_edges - <H_C>)
    expected_cut = 0.5 * (num_edges - res.fun)
    ratio = expected_cut / MAX_CUT_VAL
    ratios.append(ratio)
    
    # Update state for next layer
    final_qc = QuantumCircuit(n_qubits)
    final_qc.prepare_state(current_state)
    for i, j in edges:
        final_qc.rzz(2 * res.x[0], i, j)
    add_reflection(final_qc, current_state, res.x[1])
    current_state = Statevector.from_instruction(final_qc)
    
    print(f"{p:<6} | {res.fun:<10.4f} | {expected_cut:<10.4f} | {ratio*100:<10.2f}%")
print("-" * 45)
print(f"Final Approximation Ratio: {ratios[-1]:.4f}")
end_time = time.perf_counter()
print(f"Final runtime minutes: {(end_time - start_time)/60}")

Layer  | Energy     | Exp. Cut   | Ratio (%) 
---------------------------------------------
1      | -2.8375    | 7.4187     | 61.82     %
2      | -4.4673    | 8.2336     | 68.61     %
3      | -4.7868    | 8.3934     | 69.95     %
4      | -6.3491    | 9.1745     | 76.45     %
5      | -6.9344    | 9.4672     | 78.89     %
6      | -7.9003    | 9.9502     | 82.92     %
7      | -8.4218    | 10.2109    | 85.09     %
8      | -8.4686    | 10.2343    | 85.29     %
9      | -8.9665    | 10.4833    | 87.36     %
10     | -11.4480   | 11.7240    | 97.70     %
---------------------------------------------
Final Approximation Ratio: 0.9770
Final runtime minutes: 5.639694686117097


In [10]:
import numpy as np
import networkx as nx
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize
import time

# --- 1. Problem Setup ---
seed = np.random.seed(42)
G = nx.random_regular_graph(3, 8, seed)
n_qubits = G.number_of_nodes()
edges = G.edges()
num_edges = len(edges)

# Calculate Max Cut via Brute Force for the Ratio denominator
def get_max_cut(n, edges):
    max_c = 0
    for i in range(2**n):
        b = format(i, f'0{n}b')
        cut = sum(1 for u, v in edges if b[u] != b[v])
        if cut > max_c: max_c = cut
    return max_c

MAX_CUT_VAL = get_max_cut(n_qubits, edges)

# Define Cost Hamiltonian H_C = sum(ZiZj)
pauli_list = []
for i, j in edges:
    p_str = ["I"] * n_qubits
    p_str[n_qubits - 1 - i] = "Z"
    p_str[n_qubits - 1 - j] = "Z"
    pauli_list.append("".join(p_str))
cost_h = SparsePauliOp(pauli_list, coeffs=[1.0] * num_edges)

# --- 2. Adaptive Reflection Logic ---
def add_reflection(qc, state_vec, beta):
    n = qc.num_qubits
    v_gate = QuantumCircuit(n)
    v_gate.prepare_state(state_vec)
    
    # Unitary: V * (I - (1-e^-ib)|0><0|) * V_adj
    qc.append(v_gate.inverse(), range(n))
    qc.x(range(n))
    qc.mcp(-beta, list(range(n-1)), n-1)
    qc.x(range(n))
    qc.append(v_gate, range(n))

# --- 3. Iterative Optimization Loop ---
p_total = 10  # Number of iterative layers
estimator = StatevectorEstimator()
current_state = Statevector.from_label('+' * n_qubits)
ratios = []

print(f"{'Layer':<6} | {'Energy':<10} | {'Exp. Cut':<10} | {'Ratio (%)':<10}")
print("-" * 45)
start_time = time.perf_counter()
for p in range(1, p_total + 1):
    
    def objective(params):
        gamma, beta = params
        qc = QuantumCircuit(n_qubits)
        qc.prepare_state(current_state)
        
        # Phase Separation (Cost)
        for i, j in edges:
            qc.rzz(2 * gamma, i, j)
            
        # Adaptive Mixing (Projector Mixer)
        add_reflection(qc, current_state, beta)
        
        pub = (qc, cost_h)
        return estimator.run([pub]).result()[0].data.evs

    # Layer-wise optimization
    res = minimize(objective, [0.71, 0.39], method='COBYQA')
    
    # --- Calculate Ratio ---
    # Expected Cut = 0.5 * (num_edges - <H_C>)
    expected_cut = 0.5 * (num_edges - res.fun)
    ratio = expected_cut / MAX_CUT_VAL
    ratios.append(ratio)
    
    # Update state for next layer
    final_qc = QuantumCircuit(n_qubits)
    final_qc.prepare_state(current_state)
    for i, j in edges:
        final_qc.rzz(2 * res.x[0], i, j)
    add_reflection(final_qc, current_state, res.x[1])
    current_state = Statevector.from_instruction(final_qc)
    
    print(f"{p:<6} | {res.fun:<10.4f} | {expected_cut:<10.4f} | {ratio*100:<10.2f}%")
print("-" * 45)
print(f"Final Approximation Ratio: {ratios[-1]:.4f}")
end_time = time.perf_counter()
print(f"Final runtime minutes: {(end_time - start_time)/60}")

Layer  | Energy     | Exp. Cut   | Ratio (%) 
---------------------------------------------
1      | -2.6842    | 7.3421     | 73.42     %
2      | -4.5575    | 8.2788     | 82.79     %
3      | -5.9952    | 8.9976     | 89.98     %
4      | -7.0494    | 9.5247     | 95.25     %
5      | -7.6297    | 9.8148     | 98.15     %
6      | -7.8635    | 9.9318     | 99.32     %
7      | -7.9351    | 9.9675     | 99.68     %
8      | -7.9577    | 9.9789     | 99.79     %
9      | -7.9642    | 9.9821     | 99.82     %
10     | -7.9855    | 9.9928     | 99.93     %
---------------------------------------------
Final Approximation Ratio: 0.9993
Final runtime minutes: 1.2292955402668062


In [15]:
import numpy as np
import networkx as nx
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize
import time

# --- 1. Problem Setup ---
seed = np.random.seed(42)
G = nx.random_regular_graph(3, 8, seed)
n_qubits = G.number_of_nodes()
edges = G.edges()
num_edges = len(edges)

# Calculate Max Cut via Brute Force for the Ratio denominator
def get_max_cut(n, edges):
    max_c = 0
    for i in range(2**n):
        b = format(i, f'0{n}b')
        cut = sum(1 for u, v in edges if b[u] != b[v])
        if cut > max_c: max_c = cut
    return max_c

MAX_CUT_VAL = get_max_cut(n_qubits, edges)

# Define Cost Hamiltonian H_C = sum(ZiZj)
pauli_list = []
for i, j in edges:
    p_str = ["I"] * n_qubits
    p_str[n_qubits - 1 - i] = "Z"
    p_str[n_qubits - 1 - j] = "Z"
    pauli_list.append("".join(p_str))
cost_h = SparsePauliOp(pauli_list, coeffs=[1.0] * num_edges)

# --- 2. Adaptive Reflection Logic ---
def add_reflection(qc, state_vec, beta):
    n = qc.num_qubits
    v_gate = QuantumCircuit(n)
    v_gate.prepare_state(state_vec)
    
    # Unitary: V * (I - (1-e^-ib)|0><0|) * V_adj
    qc.append(v_gate.inverse(), range(n))
    qc.x(range(n))
    qc.mcp(-beta, list(range(n-1)), n-1)
    qc.x(range(n))
    qc.append(v_gate, range(n))

# --- 3. Iterative Optimization Loop ---
p_total = 10  # Number of iterative layers
estimator = StatevectorEstimator()
current_state = Statevector.from_label('+' * n_qubits)
ratios = []

print(f"{'Layer':<6} | {'Energy':<10} | {'Exp. Cut':<10} | {'Ratio (%)':<10}")
print("-" * 45)
start_time = time.perf_counter()
for p in range(1, p_total + 1):
    
    def objective(params):
        gamma, beta = params
        qc = QuantumCircuit(n_qubits)
        qc.prepare_state(current_state, normalize=True)
        
        # Phase Separation (Cost)
        for i, j in edges:
            qc.rzz(2 * gamma, i, j)
            
        # Adaptive Mixing (Projector Mixer)
        add_reflection(qc, current_state, beta)
        
        pub = (qc, cost_h)
        return estimator.run([pub]).result()[0].data.evs

    # Layer-wise optimization
    res = minimize(objective, [0.71, 0.39], method='BFGS')
    
    # --- Calculate Ratio ---
    # Expected Cut = 0.5 * (num_edges - <H_C>)
    expected_cut = 0.5 * (num_edges - res.fun)
    ratio = expected_cut / MAX_CUT_VAL
    ratios.append(ratio)
    
    # Update state for next layer
    final_qc = QuantumCircuit(n_qubits)
    final_qc.prepare_state(current_state)
    for i, j in edges:
        final_qc.rzz(2 * res.x[0], i, j)
    add_reflection(final_qc, current_state, res.x[1])
    current_state = Statevector.from_instruction(final_qc)
    
    print(f"{p:<6} | {res.fun:<10.4f} | {expected_cut:<10.4f} | {ratio*100:<10.2f}%")
print("-" * 45)
print(f"Final Approximation Ratio: {ratios[-1]:.4f}")
end_time = time.perf_counter()
print(f"Final runtime minutes: {(end_time - start_time)/60}")

Layer  | Energy     | Exp. Cut   | Ratio (%) 
---------------------------------------------
1      | -0.0182    | 6.0091     | 60.09     %
2      | -0.0183    | 6.0092     | 60.09     %
3      | -2.2268    | 7.1134     | 71.13     %
4      | -2.2794    | 7.1397     | 71.40     %
5      | -2.3370    | 7.1685     | 71.68     %
6      | -4.0978    | 8.0489     | 80.49     %
7      | -5.5724    | 8.7862     | 87.86     %
8      | -5.7268    | 8.8634     | 88.63     %
9      | -6.8721    | 9.4360     | 94.36     %
10     | -7.2727    | 9.6363     | 96.36     %
---------------------------------------------
Final Approximation Ratio: 0.9636
Final runtime minutes: 1.5349669034830489


In [6]:
import numpy as np
import networkx as nx
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize
import time

# --- 1. Problem Setup ---
seed = np.random.seed(42)
G = nx.random_regular_graph(3, 6, seed)
n_qubits = G.number_of_nodes()
edges = G.edges()
num_edges = len(edges)

# Calculate Max Cut via Brute Force for the Ratio denominator
def get_max_cut(n, edges):
    max_c = 0
    for i in range(2**n):
        b = format(i, f'0{n}b')
        cut = sum(1 for u, v in edges if b[u] != b[v])
        if cut > max_c: max_c = cut
    return max_c

MAX_CUT_VAL = get_max_cut(n_qubits, edges)

# Define Cost Hamiltonian H_C = sum(ZiZj)
pauli_list = []
for i, j in edges:
    p_str = ["I"] * n_qubits
    p_str[n_qubits - 1 - i] = "Z"
    p_str[n_qubits - 1 - j] = "Z"
    pauli_list.append("".join(p_str))
cost_h = SparsePauliOp(pauli_list, coeffs=[1.0] * num_edges)

# --- 2. Adaptive Reflection Logic ---
def add_reflection(qc, state_vec, beta):
    n = qc.num_qubits
    v_gate = QuantumCircuit(n)
    v_gate.prepare_state(state_vec)
    
    # Unitary: V * (I - (1-e^-ib)|0><0|) * V_adj
    qc.append(v_gate.inverse(), range(n))
    qc.x(range(n))
    qc.mcp(-beta, list(range(n-1)), n-1)
    qc.x(range(n))
    qc.append(v_gate, range(n))

# --- 3. Iterative Optimization Loop ---
p_total = 10  # Number of iterative layers
estimator = StatevectorEstimator()
current_state = Statevector.from_label('+' * n_qubits)
ratios = []

print(f"{'Layer':<6} | {'Energy':<10} | {'Exp. Cut':<10} | {'Ratio (%)':<10}")
print("-" * 45)
start_time = time.perf_counter()
for p in range(1, p_total + 1):
    
    def objective(params):
        gamma, beta = params
        qc = QuantumCircuit(n_qubits)
        qc.prepare_state(current_state, normalize=True)
        
        # Phase Separation (Cost)
        for i, j in edges:
            qc.rzz(2 * gamma, i, j)
            
        # Adaptive Mixing (Projector Mixer)
        add_reflection(qc, current_state, beta)
        
        pub = (qc, cost_h)
        return estimator.run([pub]).result()[0].data.evs

    # Layer-wise optimization
    res = minimize(objective, [0.71, 0.39], method='COBYQA')
    
    # --- Calculate Ratio ---
    # Expected Cut = 0.5 * (num_edges - <H_C>)
    expected_cut = 0.5 * (num_edges - res.fun)
    ratio = expected_cut / MAX_CUT_VAL
    ratios.append(ratio)
    
    # Update state for next layer
    final_qc = QuantumCircuit(n_qubits)
    final_qc.prepare_state(current_state)
    for i, j in edges:
        final_qc.rzz(2 * res.x[0], i, j)
    add_reflection(final_qc, current_state, res.x[1])
    current_state = Statevector.from_instruction(final_qc)
    
    print(f"{p:<6} | {res.fun:<10.4f} | {expected_cut:<10.4f} | {ratio*100:<10.2f}%")
print("-" * 45)
print(f"Final Approximation Ratio: {ratios[-1]:.4f}")
end_time = time.perf_counter()
print(f"Final runtime: {(end_time - start_time)/60} min")

Layer  | Energy     | Exp. Cut   | Ratio (%) 
---------------------------------------------
1      | -2.1766    | 5.5883     | 79.83     %
2      | -3.3424    | 6.1712     | 88.16     %
3      | -4.2657    | 6.6328     | 94.75     %
4      | -4.7326    | 6.8663     | 98.09     %
5      | -4.9662    | 6.9831     | 99.76     %
6      | -4.9962    | 6.9981     | 99.97     %
7      | -4.9987    | 6.9993     | 99.99     %
8      | -4.9994    | 6.9997     | 100.00    %
9      | -4.9997    | 6.9999     | 100.00    %
10     | -5.0000    | 7.0000     | 100.00    %
---------------------------------------------
Final Approximation Ratio: 1.0000
Final runtime: 0.34613079166641303 min


Our analytical resutls for the optimal parameters for QAOA-1 in the Gaussian limit are:\
$\beta^* = -\pi/2$ and $\gamma^* = \frac{2}{\sqrt{2|E|}}$\
where $|E|$ is the number of edges in the graph. 

In [19]:
import numpy as np
import networkx as nx
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize
import time

# --- 1. Problem Setup ---
seed = np.random.seed(42)
G = nx.random_regular_graph(3, 8, seed)
n_qubits = G.number_of_nodes()
edges = G.edges()
num_edges = len(edges)

# Calculate Max Cut via Brute Force for the Ratio denominator
def get_max_cut(n, edges):
    max_c = 0
    for i in range(2**n):
        b = format(i, f'0{n}b')
        cut = sum(1 for u, v in edges if b[u] != b[v])
        if cut > max_c: max_c = cut
    return max_c

MAX_CUT_VAL = get_max_cut(n_qubits, edges)

# Define Cost Hamiltonian H_C = sum(ZiZj)
pauli_list = []
for i, j in edges:
    p_str = ["I"] * n_qubits
    p_str[n_qubits - 1 - i] = "Z"
    p_str[n_qubits - 1 - j] = "Z"
    pauli_list.append("".join(p_str))
cost_h = SparsePauliOp(pauli_list, coeffs=[1.0] * num_edges)

# --- 2. Adaptive Reflection Logic ---
def add_reflection(qc, state_vec, beta):
    n = qc.num_qubits
    v_gate = QuantumCircuit(n)
    v_gate.prepare_state(state_vec)
    
    # Unitary: V * (I - (1-e^-ib)|0><0|) * V_adj
    qc.append(v_gate.inverse(), range(n))
    qc.x(range(n))
    qc.mcp(-beta, list(range(n-1)), n-1)
    qc.x(range(n))
    qc.append(v_gate, range(n))

# --- 3. Iterative Optimization Loop ---
p_total = n_qubits + 2  # Number of iterative layers of order n
estimator = StatevectorEstimator()
current_state = Statevector.from_label('+' * n_qubits)
ratios = []

print(f"{'Layer':<6} | {'Energy':<10} | {'Exp. Cut':<10} | {'Ratio (%)':<10}")
print("-" * 45)
start_time = time.perf_counter()
for p in range(1, p_total + 1):
    
    def objective(params):
        gamma, beta = params
        qc = QuantumCircuit(n_qubits)
        qc.prepare_state(current_state, normalize=True)
        
        # Phase Separation (Cost)
        for i, j in edges:
            qc.rzz(2 * gamma, i, j)
            
        # Adaptive Mixing (Projector Mixer)
        add_reflection(qc, current_state, beta)
        
        pub = (qc, cost_h)
        return estimator.run([pub]).result()[0].data.evs

    # Layer-wise optimization
    res = minimize(objective, [2 / np.sqrt(2 * num_edges), -np.pi / 2], method='COBYLA')
    # --- Calculate Ratio ---
    # Expected Cut = 0.5 * (num_edges - <H_C>)
    expected_cut = 0.5 * (num_edges - objective(params))
    ratio = expected_cut / MAX_CUT_VAL
    ratios.append(ratio)
    
    # Update state for next layer
    final_qc = QuantumCircuit(n_qubits)
    final_qc.prepare_state(current_state)
    for i, j in edges:
        final_qc.rzz(2 * res.x[0], i, j)
    add_reflection(final_qc, current_state, res.x[1])
    current_state = Statevector.from_instruction(final_qc)
    
    print(f"{p:<6} | {res.fun:<10.4f} | {expected_cut:<10.4f} | {ratio*100:<10.2f}%")
print("-" * 45)
print(f"Final Approximation Ratio: {ratios[-1]:.4f}")
end_time = time.perf_counter()
print(f"Final runtime: {(end_time - start_time)/60} min")

Layer  | Energy     | Exp. Cut   | Ratio (%) 
---------------------------------------------
1      | -2.6842    | 6.5472     | 65.47     %
2      | -4.5551    | 8.2506     | 82.51     %
3      | -5.9931    | 7.9455     | 79.45     %
4      | -7.0477    | 9.4962     | 94.96     %
5      | -7.1174    | 9.3306     | 93.31     %
6      | -7.2849    | 9.3410     | 93.41     %
7      | -7.7909    | 8.9850     | 89.85     %
8      | -7.9068    | 9.8798     | 98.80     %
9      | -7.9754    | 9.8604     | 98.60     %
10     | -7.9848    | 9.9854     | 99.85     %
---------------------------------------------
Final Approximation Ratio: 0.9985
Final runtime: 3.584890879850718 min


In [31]:
import numpy as np
import networkx as nx
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize
import time

# --- 1. Problem Setup ---
seed = np.random.seed(42)
G = nx.random_regular_graph(3, 8, seed)
n_qubits = G.number_of_nodes()
edges = G.edges()
num_edges = len(edges)

# Calculate Max Cut via Brute Force for the Ratio denominator
def get_max_cut(n, edges):
    max_c = 0
    for i in range(2**n):
        b = format(i, f'0{n}b')
        cut = sum(1 for u, v in edges if b[u] != b[v])
        if cut > max_c: max_c = cut
    return max_c

MAX_CUT_VAL = get_max_cut(n_qubits, edges)

# Define Cost Hamiltonian H_C = sum(ZiZj)
pauli_list = []
for i, j in edges:
    p_str = ["I"] * n_qubits
    p_str[n_qubits - 1 - i] = "Z"
    p_str[n_qubits - 1 - j] = "Z"
    pauli_list.append("".join(p_str))
cost_h = SparsePauliOp(pauli_list, coeffs=[1.0] * num_edges)

# --- 2. Adaptive Reflection Logic ---
def add_reflection(qc, state_vec, beta):
    n = qc.num_qubits
    v_gate = QuantumCircuit(n)
    v_gate.prepare_state(state_vec)
    
    # Unitary: V * (I - (1-e^-ib)|0><0|) * V_adj
    qc.append(v_gate.inverse(), range(n))
    qc.x(range(n))
    qc.mcp(-beta, list(range(n-1)), n-1)
    qc.x(range(n))
    qc.append(v_gate, range(n))

# --- 3. Iterative Optimization Loop ---
p_total = n_qubits + 2  # Number of iterative layers of order n
estimator = StatevectorEstimator()
current_state = Statevector.from_label('+' * n_qubits)
ratios = []

print(f"{'Layer':<6} | {'Energy':<10} | {'Exp. Cut':<10} | {'Ratio (%)':<10}")
print("-" * 45)
start_time = time.perf_counter()
for p in range(1, p_total + 1):
    
    def objective(params):
        gamma, beta = params
        qc = QuantumCircuit(n_qubits)
        qc.prepare_state(current_state, normalize=True)
        
        # Phase Separation (Cost)
        for i, j in edges:
            qc.rzz(2 * gamma, i, j)
            
        # Adaptive Mixing (Projector Mixer)
        add_reflection(qc, current_state, beta)
        
        pub = (qc, cost_h)
        return estimator.run([pub]).result()[0].data.evs

    # Layer-wise optimization
    res = minimize(objective, [np.sqrt(2 / num_edges), -np.pi / 2], method='COBYQA')
    # --- Calculate Ratio ---
    # Expected Cut = 0.5 * (num_edges - <H_C>)
    expected_cut = 0.5 * (num_edges - objective(params))
    ratio = expected_cut / MAX_CUT_VAL
    ratios.append(ratio)
    
    # Update state for next layer
    final_qc = QuantumCircuit(n_qubits)
    final_qc.prepare_state(current_state)
    for i, j in edges:
        final_qc.rzz(2 * res.x[0], i, j)
    add_reflection(final_qc, current_state, res.x[1])
    current_state = Statevector.from_instruction(final_qc)
    
    print(f"{p:<6} | {res.fun:<10.4f} | {expected_cut:<10.4f} | {ratio*100:<10.2f}%")
print("-" * 45)
print(f"Final Approximation Ratio: {ratios[-1]:.4f}")
end_time = time.perf_counter()
print(f"Final runtime: {(end_time - start_time)/60} min")

Layer  | Energy     | Exp. Cut   | Ratio (%) 
---------------------------------------------
1      | -2.9656    | 6.8001     | 68.00     %
2      | -4.8094    | 8.4023     | 84.02     %
3      | -4.8838    | 7.7256     | 77.26     %
4      | -6.5101    | 7.9199     | 79.20     %
5      | -7.3001    | 9.5186     | 95.19     %
6      | -7.7325    | 9.2664     | 92.66     %
7      | -7.9267    | 9.9309     | 99.31     %
8      | -7.9374    | 9.9120     | 99.12     %
9      | -7.9896    | 9.9119     | 99.12     %
10     | -7.9960    | 9.9868     | 99.87     %
---------------------------------------------
Final Approximation Ratio: 0.9987
Final runtime: 1.369814291667232 min


In [28]:
import numpy as np
import networkx as nx
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize
import time

# --- 1. Problem Setup ---
seed = np.random.seed(42)
G = nx.random_regular_graph(3, 10, seed)
n_qubits = G.number_of_nodes()
edges = G.edges()
num_edges = len(edges)

# Calculate Max Cut via Brute Force for the Ratio denominator
def get_max_cut(n, edges):
    max_c = 0
    for i in range(2**n):
        b = format(i, f'0{n}b')
        cut = sum(1 for u, v in edges if b[u] != b[v])
        if cut > max_c: max_c = cut
    return max_c

MAX_CUT_VAL = get_max_cut(n_qubits, edges)

# Define Cost Hamiltonian H_C = sum(ZiZj)
pauli_list = []
for i, j in edges:
    p_str = ["I"] * n_qubits
    p_str[n_qubits - 1 - i] = "Z"
    p_str[n_qubits - 1 - j] = "Z"
    pauli_list.append("".join(p_str))
cost_h = SparsePauliOp(pauli_list, coeffs=[1.0] * num_edges)

# --- 2. Adaptive Reflection Logic ---
def add_reflection(qc, state_vec, beta):
    n = qc.num_qubits
    v_gate = QuantumCircuit(n)
    v_gate.prepare_state(state_vec)
    
    # Unitary: V * (I - (1-e^-ib)|0><0|) * V_adj
    qc.append(v_gate.inverse(), range(n))
    qc.x(range(n))
    qc.mcp(-beta, list(range(n-1)), n-1)
    qc.x(range(n))
    qc.append(v_gate, range(n))

# --- 3. Iterative Optimization Loop ---
p_total = n_qubits + 2  # Number of iterative layers of order n
estimator = StatevectorEstimator()
current_state = Statevector.from_label('+' * n_qubits)
ratios = []

print(f"{'Layer':<6} | {'Energy':<10} | {'Exp. Cut':<10} | {'Ratio (%)':<10}")
print("-" * 45)
start_time = time.perf_counter()
for p in range(1, p_total + 1):
    
    def objective(params):
        gamma, beta = params
        qc = QuantumCircuit(n_qubits)
        qc.prepare_state(current_state, normalize=True)
        
        # Phase Separation (Cost)
        for i, j in edges:
            qc.rzz(2 * gamma, i, j)
            
        # Adaptive Mixing (Projector Mixer)
        add_reflection(qc, current_state, beta)
        
        pub = (qc, cost_h)
        return estimator.run([pub]).result()[0].data.evs

    # Layer-wise optimization
    res = minimize(objective, [np.sqrt(2 / num_edges), -np.pi / 2], method='COBYLA')
    # --- Calculate Ratio ---
    # Expected Cut = 0.5 * (num_edges - <H_C>)
    expected_cut = 0.5 * (num_edges - objective(params))
    ratio = expected_cut / MAX_CUT_VAL
    ratios.append(ratio)
    
    # Update state for next layer
    final_qc = QuantumCircuit(n_qubits)
    final_qc.prepare_state(current_state)
    for i, j in edges:
        final_qc.rzz(2 * res.x[0], i, j)
    add_reflection(final_qc, current_state, res.x[1])
    current_state = Statevector.from_instruction(final_qc)
    
    print(f"{p:<6} | {res.fun:<10.4f} | {expected_cut:<10.4f} | {ratio*100:<10.2f}%")
print("-" * 45)
print(f"Final Approximation Ratio: {ratios[-1]:.4f}")
end_time = time.perf_counter()
print(f"Final runtime: {(end_time - start_time)/60} min")

Layer  | Energy     | Exp. Cut   | Ratio (%) 
---------------------------------------------
1      | -3.2185    | 8.0094     | 61.61     %


KeyboardInterrupt: 

In [17]:
import numpy as np
import networkx as nx
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize
import time

# --- 1. Problem Setup ---
seed = np.random.seed(42)
G = nx.random_regular_graph(3, 12, seed)
n_qubits = G.number_of_nodes()
edges = G.edges()
num_edges = len(edges)

# Calculate Max Cut via Brute Force for the Ratio denominator
def get_max_cut(n, edges):
    max_c = 0
    for i in range(2**n):
        b = format(i, f'0{n}b')
        cut = sum(1 for u, v in edges if b[u] != b[v])
        if cut > max_c: max_c = cut
    return max_c

MAX_CUT_VAL = get_max_cut(n_qubits, edges)

# Define Cost Hamiltonian H_C = sum(ZiZj)
pauli_list = []
for i, j in edges:
    p_str = ["I"] * n_qubits
    p_str[n_qubits - 1 - i] = "Z"
    p_str[n_qubits - 1 - j] = "Z"
    pauli_list.append("".join(p_str))
cost_h = SparsePauliOp(pauli_list, coeffs=[1.0] * num_edges)

# --- 2. Adaptive Reflection Logic ---
def add_reflection(qc, state_vec, beta):
    n = qc.num_qubits
    v_gate = QuantumCircuit(n)
    v_gate.prepare_state(state_vec)
    
    # Unitary: V * (I - (1-e^-ib)|0><0|) * V_adj
    qc.append(v_gate.inverse(), range(n))
    qc.x(range(n))
    qc.mcp(-beta, list(range(n-1)), n-1)
    qc.x(range(n))
    qc.append(v_gate, range(n))

# --- 3. Iterative Optimization Loop ---
p_total = n_qubits + 2  # Number of iterative layers of order n
estimator = StatevectorEstimator()
current_state = Statevector.from_label('+' * n_qubits)
ratios = []

print(f"{'Layer':<6} | {'Energy':<10} | {'Exp. Cut':<10} | {'Ratio (%)':<10}")
print("-" * 45)
start_time = time.perf_counter()
for p in range(1, p_total + 1):
    
    def objective(params):
        gamma, beta = params
        qc = QuantumCircuit(n_qubits)
        qc.prepare_state(current_state, normalize=True)
        
        # Phase Separation (Cost)
        for i, j in edges:
            qc.rzz(2 * gamma, i, j)
            
        # Adaptive Mixing (Projector Mixer)
        add_reflection(qc, current_state, beta)
        
        pub = (qc, cost_h)
        return estimator.run([pub]).result()[0].data.evs

    # Layer-wise optimization
    res = minimize(objective, [np.sqrt(2 / num_edges), -np.pi / 2], method='COBYLA')
    # --- Calculate Ratio ---
    # Expected Cut = 0.5 * (num_edges - <H_C>)
    expected_cut = 0.5 * (num_edges - objective(params))
    ratio = expected_cut / MAX_CUT_VAL
    ratios.append(ratio)
    
    # Update state for next layer
    final_qc = QuantumCircuit(n_qubits)
    final_qc.prepare_state(current_state)
    for i, j in edges:
        final_qc.rzz(2 * res.x[0], i, j)
    add_reflection(final_qc, current_state, res.x[1])
    current_state = Statevector.from_instruction(final_qc)
    
    print(f"{p:<6} | {res.fun:<10.4f} | {expected_cut:<10.4f} | {ratio*100:<10.2f}%")
print("-" * 45)
print(f"Final Approximation Ratio: {ratios[-1]:.4f}")
end_time = time.perf_counter()
print(f"Final runtime: {(end_time - start_time)/60} min")

Layer  | Energy     | Exp. Cut   | Ratio (%) 
---------------------------------------------
1      | -3.4688    | 9.3367     | 62.24     %


KeyboardInterrupt: 